# LifeStack GRPO Training on Colab

Use this notebook with a Colab GPU runtime. It installs Unsloth + TRL, verifies the environment, applies the TRL/PEFT compatibility patch if needed, runs a dry-run, then starts curriculum training.

Recommended runtime: **T4 or better GPU**.

In [ ]:
# 1) GPU sanity check
import subprocess
import sys
import platform

subprocess.run(['nvidia-smi'], check=False)
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# 2) Get the repo into /content/Meta-R2
# If you uploaded this notebook from the repo, this still works: it reuses the folder if present.
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/Meta-R2')
REPO_URL = 'https://github.com/oki-dokii/Meta-R2.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repo already exists: {REPO_DIR}')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# 3) Install training dependencies
# Keep this runtime training-only. Installing the full demo/RAG stack first can break
# Colab's NumPy/SciPy/sklearn binary compatibility.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# Stable numeric stack for Python 3.12 Colab. Force reinstall fixes errors like:
# ImportError: cannot import name '_center' from numpy._core.umath
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2',
], check=True)

# Project packages needed by train_trl.py and reward/memory helpers.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'pydantic', 'openenv-core', 'chromadb', 'sentence-transformers', 'gymnasium',
], check=True)

# GRPO/LoRA training stack.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'unsloth', 'trl', 'transformers', 'peft', 'accelerate', 'datasets',
    'bitsandbytes', 'tensorboard', 'huggingface_hub',
], check=True)

# Re-pin numeric packages after dependency resolution so later installs do not leave mixed wheels.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2',
], check=True)

print('Install complete. If the next import cell still fails, restart runtime and rerun from cell 1.')

In [ ]:
# 4) Optional Hugging Face token for faster downloads and pushing checkpoints
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN found. Training can still run, but downloads may be rate-limited.')
except Exception as exc:
    print('Colab Secrets unavailable:', exc)

In [ ]:
# 5) Verify imports and versions
# If this cell fails immediately after installs, use Runtime > Restart runtime,
# then rerun cells from the top. Colab can keep old binary modules loaded.
import numpy
import scipy
import sklearn
import torch
import transformers
import trl
import peft
import accelerate
import datasets
import bitsandbytes
import unsloth

print('numpy:', numpy.__version__)
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('peft:', peft.__version__)
print('accelerate:', accelerate.__version__)
print('datasets:', datasets.__version__)
print('unsloth import: OK')

In [ ]:
# 6) Apply TRL/PEFT compatibility patch if the cloned script does not have it yet
import subprocess
import sys
from pathlib import Path

train_path = Path('/content/Meta-R2/scripts/train_trl.py')
text = train_path.read_text()
original = text

if '_ensure_trl_model_compat' not in text:
    anchor = "def _tensorboard_available() -> bool:\n    try:\n        import tensorboard  # noqa: F401\n        return True\n    except ImportError:\n        return False\n"
    helper = anchor + "\n\ndef _ensure_trl_model_compat(model):\n    if not hasattr(model, 'warnings_issued'):\n        model.warnings_issued = {}\n    return model\n"
    if anchor not in text:
        raise RuntimeError('Could not find _tensorboard_available anchor in train_trl.py')
    text = text.replace(anchor, helper)

full_anchor = 'model, tokenizer = load_model()\n\n    # ── Determine where to start'
if full_anchor in text:
    text = text.replace(
        full_anchor,
        'model, tokenizer = load_model()\n    model = _ensure_trl_model_compat(model)\n\n    # ── Determine where to start',
    )

dry_anchor = 'model, tokenizer = load_model_for_dry_run()\n\n    dataset = generate_dataset'
if dry_anchor in text:
    text = text.replace(
        dry_anchor,
        'model, tokenizer = load_model_for_dry_run()\n    model = _ensure_trl_model_compat(model)\n\n    dataset = generate_dataset',
    )

if text != original:
    train_path.write_text(text)
    print('Patched train_trl.py for TRL/PEFT warnings_issued compatibility.')
else:
    print('Compatibility patch already present.')

subprocess.run([sys.executable, '-m', 'py_compile', str(train_path)], check=True)

In [ ]:
# 7) Dry-run: quick pipeline check before spending GPU time
import subprocess
import sys

subprocess.run([sys.executable, '/content/Meta-R2/scripts/train_trl.py', '--dry-run'], check=True)

In [ ]:
# 8) Full curriculum training
# Start small first. Increase PROMPTS_PER_STAGE after this works.
import subprocess
import sys

STAGES = 3
PROMPTS_PER_STAGE = 100
OUTPUT_DIR = '/content/Meta-R2/lifestack_model'

subprocess.run([
    sys.executable,
    '/content/Meta-R2/scripts/train_trl.py',
    '--stages', str(STAGES),
    '--prompts-per-stage', str(PROMPTS_PER_STAGE),
    '--output-dir', OUTPUT_DIR,
], check=True)

In [ ]:
# 9) Resume after a Colab disconnect or runtime restart
import subprocess
import sys

STAGES = 3
PROMPTS_PER_STAGE = 100
OUTPUT_DIR = '/content/Meta-R2/lifestack_model'

subprocess.run([
    sys.executable,
    '/content/Meta-R2/scripts/train_trl.py',
    '--resume',
    '--stages', str(STAGES),
    '--prompts-per-stage', str(PROMPTS_PER_STAGE),
    '--output-dir', OUTPUT_DIR,
], check=True)

---
## 10 · Training Evidence — Plots from the Real Run

These cells re-generate the reward and loss plots from the saved log produced during the actual v4 training run.  
The log (`train_run_v4.log`) and the pre-generated summary (`grpo_reward_curve_v4.png`) are committed to the repository, so judges can inspect them without re-running training.

> **To reproduce from scratch**: run cells 7–8 above, then come back here.  
> **To just view evidence**: the cells below load the committed artefacts directly.

In [ ]:
# 10-a) Ensure matplotlib is available (already present in Colab, but install just in case)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'], check=True)
print('matplotlib ready')

In [ ]:
# 10-b) Generate all training plots from train_run_v4.log
# The script parses both the reward-step lines AND HF Trainer dict lines.
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/Meta-R2')
LOG_FILE = REPO / 'train_run_v4.log'
OUT_DIR  = REPO / 'plots'

if not LOG_FILE.exists():
    print(f'WARNING: {LOG_FILE} not found. Have you run the training cell (step 8)?')
    print('Showing the pre-committed artefact instead (cell 10-c below).')
else:
    result = subprocess.run(
        [sys.executable, str(REPO / 'scripts' / 'plot_training.py'),
         '--log', str(LOG_FILE),
         '--out', str(OUT_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

In [ ]:
# 10-c) Display training evidence inline
from pathlib import Path
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

REPO = Path('/content/Meta-R2')

# Priority order: freshly generated > pre-committed v4 artefact
candidates = [
    ('Training Summary (4-panel)',       REPO / 'plots' / 'training_summary.png'),
    ('Reward Curve',                     REPO / 'plots' / 'reward_curve.png'),
    ('Reward Components',                REPO / 'plots' / 'reward_components.png'),
    ('Loss Curve',                       REPO / 'plots' / 'loss_curve.png'),
    ('Pre-committed Reward Curve (v4)',  REPO / 'grpo_reward_curve_v4.png'),
]

shown = 0
for title, p in candidates:
    if p.exists():
        print(f'--- {title} ---')
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=12, pad=8)
        plt.tight_layout()
        plt.show()
        shown += 1

if shown == 0:
    print('No plot files found. Run the training (cells 7–8) then re-run cell 10-b.')

In [ ]:
# 10-d) Print training statistics summary
import re, json
from pathlib import Path

REPO = Path('/content/Meta-R2')
LOG_FILE = REPO / 'train_run_v4.log'

if not LOG_FILE.exists():
    print('train_run_v4.log not found – run training first.')
else:
    text = LOG_FILE.read_text(errors='replace')

    # Parse reward lines
    STEP_RE = re.compile(
        r'\[step\s+(\d+)\]\s+r0=([+-]?\d+\.\d+).*?r_lt=([+-]?\d+\.\d+)'
    )
    matches = STEP_RE.findall(text)

    if matches:
        import numpy as np
        steps   = [int(m[0])   for m in matches]
        rewards = [float(m[1]) for m in matches]
        ltr     = [float(m[2]) for m in matches]
        n = len(rewards)
        q1, q2 = int(n * 0.1), int(n * 0.9)
        print('=' * 52)
        print(f'  LifeStack GRPO  –  Run v4  Training Summary')
        print('=' * 52)
        print(f'  Reward steps logged : {n:>8,}')
        print(f'  Step range          : {steps[0]:>8,} – {steps[-1]:,}')
        print(f'  Mean r0  (all)      : {np.mean(rewards):>8.4f}')
        print(f'  Mean r0  (first 10%): {np.mean(rewards[:q1 or 1]):>8.4f}')
        print(f'  Mean r0  (last  10%): {np.mean(rewards[q2:]):>8.4f}')
        print(f'  Improvement Δ       : {np.mean(rewards[q2:]) - np.mean(rewards[:q1 or 1]):>+8.4f}')
        print(f'  Mean r_lt (all)     : {np.mean(ltr):>8.4f}')
        print(f'  Min  r0             : {min(rewards):>8.4f}')
        print(f'  Max  r0             : {max(rewards):>8.4f}')
        print('=' * 52)
    else:
        print('No [step N] r0=... lines found. Check the log format.')

    # Also parse HF Trainer logs
    DICT_RE = re.compile(r'\{[^{}]+\}')
    losses = []
    for m in DICT_RE.finditer(text):
        try:
            d = eval(m.group(0), {'__builtins__': {}})
            if isinstance(d, dict) and 'loss' in d:
                losses.append(float(d['loss']))
        except Exception:
            pass
    if losses:
        import numpy as np
        print(f'\n  Trainer loss entries: {len(losses):>8,}')
        print(f'  Loss start (avg 10%): {np.mean(losses[:max(1,len(losses)//10)]):>8.4f}')
        print(f'  Loss end   (avg 10%): {np.mean(losses[-max(1,len(losses)//10):]):>8.4f}')

## If It Fails

For the NumPy `_center` error specifically, rerun the install cell, then use **Runtime > Restart runtime**, then rerun from the top. Colab can keep old binary modules loaded after `pip` replaces NumPy.

Send the output of these commands along with the full traceback:

```bash
nvidia-smi
pip show numpy scipy scikit-learn unsloth trl transformers peft torch bitsandbytes accelerate
```

Most common causes are no GPU runtime, Unsloth not installed after a runtime reset, or a package-version mismatch between NumPy/SciPy/sklearn and TRL/PEFT/Transformers.